# Enveda CASMI 2026 — Spectrum Retrieval Baseline

This notebook predicts up to 25 candidate SMILES for each test molecule by retrieving reference spectra with a similar precursor mass and ranking them by MS/MS fragment similarity. It is designed to run on Kaggle with internet disabled; the competition data is read from `/kaggle/input`.

**Run all cells**, then confirm `/kaggle/working/submission.csv` exists and has one non-empty row per `molecule_id`. This is a transparent starter baseline, not a claim of leaderboard performance. It uses no external data and does not require a GPU.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time

COMPETITION_SLUG = "enveda-CASMI26-molecule-id-mass-spectra"
INPUT_ROOT = Path("/kaggle/input")


def find_input(filename: str) -> Path:
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} under {INPUT_ROOT}. "
            "Attach the Enveda CASMI 2026 competition data to this notebook."
        )
    return matches[0]


TRAIN_PATH = find_input("train.parquet")
TEST_PATH = find_input("test.parquet")
SAMPLE_PATH = find_input("sample_submission.csv")

print("Train:", TRAIN_PATH)
print("Test:", TEST_PATH)
print("Sample:", SAMPLE_PATH)

# Read only the columns needed by this baseline. In particular, avoid
# materializing unused metadata columns from the large training parquet.
TRAIN_COLUMNS = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "precursor_mz",
    "ms2_mzs",
    "ms2_normalized_intensities",
]
train = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
test = pd.read_parquet(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH, dtype={"molecule_id": str})

required_test = {
    "molecule_id", "adduct", "precursor_mz",
    "ms2_mzs", "ms2_normalized_intensities",
}
missing = required_test - set(test.columns)
if missing:
    raise ValueError(f"Missing test columns: {sorted(missing)}")

train = train.dropna(
    subset=["inchikey14", "normalized_smiles", "adduct", "precursor_mz"]
).copy()
train["inchikey14"] = train["inchikey14"].astype(str)
train["normalized_smiles"] = train["normalized_smiles"].astype(str)
train["adduct"] = train["adduct"].astype(str)
train["precursor_mz"] = pd.to_numeric(train["precursor_mz"], errors="coerce")
train = train[
    np.isfinite(train["precursor_mz"])
    & train["normalized_smiles"].ne("")
    & train["inchikey14"].ne("")
].reset_index(drop=True)

test["molecule_id"] = test["molecule_id"].astype(str)
test["adduct"] = test["adduct"].fillna("").astype(str)
test["precursor_mz"] = pd.to_numeric(test["precursor_mz"], errors="coerce")

assert sample_submission.columns.tolist() == ["molecule_id", "smiles"]
assert sample_submission["molecule_id"].is_unique
assert test["molecule_id"].notna().all()

print(f"Training reference spectra: {len(train):,}")
print(f"Training reference molecules: {train['inchikey14'].nunique():,}")
print(f"Test spectra: {len(test):,}")
print(f"Test molecules: {test['molecule_id'].nunique():,}")
print(f"Sample rows: {len(sample_submission):,}")

In [ ]:
# Candidate retrieval and MS/MS fragment-similarity helpers.
# Spectra are converted to sparse 0.01 Da fragment bins and compared with
# normalized cosine similarity. Precursor windows widen only as needed.

FRAGMENT_BIN_DA = 0.01
PRECURSOR_WINDOWS_PPM = (2, 5, 10, 25)
MAX_CANDIDATE_SPECTRA = 2500
MAX_GUESSES = 25

# Sorted precursor-mass indexes make candidate lookup fast.
adduct_index = {}
for adduct, group in train.groupby("adduct", sort=False):
    positions = group.index.to_numpy(dtype=np.int64)
    masses = train.loc[positions, "precursor_mz"].to_numpy(dtype=np.float64)
    order = np.argsort(masses)
    adduct_index[str(adduct)] = (masses[order], positions[order])

all_masses = train["precursor_mz"].to_numpy(dtype=np.float64)
all_positions = np.arange(len(train), dtype=np.int64)
global_order = np.argsort(all_masses)
all_masses = all_masses[global_order]
all_positions = all_positions[global_order]


def spectrum_vector(mzs, intensities):
    """Return unique sorted fragment bins and unit-length sqrt intensities."""
    try:
        mzs = np.asarray(mzs, dtype=np.float64).reshape(-1)
        intensities = np.asarray(intensities, dtype=np.float64).reshape(-1)
    except (TypeError, ValueError):
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    if len(mzs) == 0 or len(mzs) != len(intensities):
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    valid = np.isfinite(mzs) & np.isfinite(intensities) & (mzs >= 0) & (intensities > 0)
    mzs, intensities = mzs[valid], intensities[valid]
    if len(mzs) == 0:
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    bins = np.rint(mzs / FRAGMENT_BIN_DA).astype(np.int32)
    unique_bins, inverse = np.unique(bins, return_inverse=True)
    values = np.bincount(inverse, weights=np.sqrt(intensities)).astype(np.float32)
    norm = np.linalg.norm(values)
    if norm > 0:
        values /= norm
    return unique_bins, values


def fragment_similarity(query_bins, query_values, ref_bins, ref_values):
    """Sparse cosine similarity between two binned spectra."""
    if len(query_bins) == 0 or len(ref_bins) == 0:
        return 0.0
    _, query_idx, ref_idx = np.intersect1d(
        query_bins, ref_bins, assume_unique=True, return_indices=True
    )
    if len(query_idx) == 0:
        return 0.0
    return float(np.dot(query_values[query_idx], ref_values[ref_idx]))


def get_candidate_positions(adduct, precursor_mz):
    """Retrieve nearby precursor references; fall back to nearest masses."""
    if not np.isfinite(precursor_mz):
        precursor_mz = float(train["precursor_mz"].median())

    entry = adduct_index.get(str(adduct))
    positions = np.empty(0, dtype=np.int64)
    masses = None
    if entry is not None:
        masses, adduct_positions = entry
        for ppm in PRECURSOR_WINDOWS_PPM:
            delta = max(abs(float(precursor_mz)) * ppm / 1_000_000, 0.002)
            left = np.searchsorted(masses, precursor_mz - delta, side="left")
            right = np.searchsorted(masses, precursor_mz + delta, side="right")
            if right > left:
                positions = adduct_positions[left:right]
                break

    if len(positions) == 0:
        # Rare out-of-domain case: nearest precursor masses, preferring the
        # same adduct when available.
        if entry is not None and len(masses):
            nearest = np.searchsorted(masses, precursor_mz)
            lo = max(0, nearest - MAX_CANDIDATE_SPECTRA // 2)
            hi = min(len(masses), lo + MAX_CANDIDATE_SPECTRA)
            positions = entry[1][lo:hi]
        else:
            nearest = np.searchsorted(all_masses, precursor_mz)
            lo = max(0, nearest - MAX_CANDIDATE_SPECTRA // 2)
            hi = min(len(all_masses), lo + MAX_CANDIDATE_SPECTRA)
            positions = all_positions[lo:hi]

    if len(positions) > MAX_CANDIDATE_SPECTRA:
        ref_masses = train.loc[positions, "precursor_mz"].to_numpy(dtype=np.float64)
        keep = np.argpartition(
            np.abs(ref_masses - precursor_mz), MAX_CANDIDATE_SPECTRA - 1
        )[:MAX_CANDIDATE_SPECTRA]
        positions = positions[keep]
    return positions


# Smoke-test the matching helpers without running the full inference loop.
_demo_bins, _demo_values = spectrum_vector([50.0, 75.0], [1.0, 0.5])
assert fragment_similarity(_demo_bins, _demo_values, _demo_bins, _demo_values) > 0.99
assert len(get_candidate_positions(test.iloc[0]["adduct"], test.iloc[0]["precursor_mz"])) > 0
print("Index ready:", len(adduct_index), "adducts; sparse-fragment smoke test passed.")

In [ ]:
# Rank retrieved references and write the required submission.csv.

from functools import lru_cache

# Compact numpy/object arrays avoid repeated DataFrame row construction.
train_molecules = train["inchikey14"].to_numpy(dtype=object)
train_ms2_mzs = train["ms2_mzs"].to_numpy(dtype=object)
train_ms2_intensities = train["ms2_normalized_intensities"].to_numpy(dtype=object)
train_precursors = train["precursor_mz"].to_numpy(dtype=np.float64)

# One representative training SMILES for each InChIKey14.
smiles_lookup = (
    train.groupby(["inchikey14", "normalized_smiles"], sort=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["inchikey14", "count"], ascending=[True, False])
    .drop_duplicates("inchikey14")
    .set_index("inchikey14")["normalized_smiles"]
    .astype(str)
    .to_dict()
)
frequent_molecules = train["inchikey14"].value_counts().index.astype(str).tolist()

# Reuse vectorized reference spectra across queries, with a bounded cache to
# prevent the large training set from consuming all notebook memory.
@lru_cache(maxsize=50000)
def reference_vector(position):
    position = int(position)
    return spectrum_vector(train_ms2_mzs[position], train_ms2_intensities[position])

# Candidate scores are combined across all spectra belonging to a molecule_id.
score_by_molecule_id = {}
start_time = time.time()

for row_number, query in test.iterrows():
    molecule_id = str(query["molecule_id"])
    precursor_mz = float(query["precursor_mz"])
    if not np.isfinite(precursor_mz):
        precursor_mz = float(train["precursor_mz"].median())
    candidate_positions = get_candidate_positions(query["adduct"], precursor_mz)
    query_bins, query_values = spectrum_vector(
        query["ms2_mzs"], query["ms2_normalized_intensities"]
    )

    candidate_scores = score_by_molecule_id.setdefault(molecule_id, {})
    for pos in candidate_positions:
        pos = int(pos)
        ref_bins, ref_values = reference_vector(pos)
        sim = fragment_similarity(query_bins, query_values, ref_bins, ref_values)
        ppm_error = abs(train_precursors[pos] - precursor_mz) / max(abs(precursor_mz), 1e-12) * 1e6
        # Spectral evidence dominates; precursor agreement is a small tie-breaker.
        score = sim + 0.02 * np.exp(-ppm_error / 10.0)
        key = str(train_molecules[pos])
        if score > candidate_scores.get(key, -np.inf):
            candidate_scores[key] = float(score)

    if (row_number + 1) % 100 == 0 or row_number + 1 == len(test):
        elapsed = time.time() - start_time
        print(f"Processed {row_number + 1:,}/{len(test):,} spectra in {elapsed:.1f}s")

# One output row per sample ID, in sample order. The frequent-reference
# fallback is used only to fill missing/short candidate lists.
submission_rows = []
for molecule_id in sample_submission["molecule_id"].astype(str):
    ranked = sorted(
        score_by_molecule_id.get(molecule_id, {}).items(),
        key=lambda item: item[1],
        reverse=True,
    )
    guesses = []
    seen = set()
    for key, _score in ranked:
        smiles = smiles_lookup.get(key)
        if smiles and smiles not in seen:
            guesses.append(smiles)
            seen.add(smiles)
        if len(guesses) == MAX_GUESSES:
            break

    # If retrieval was empty (or yielded fewer than 25 structures), append
    # valid known-reference SMILES as lower-ranked candidates.
    if len(guesses) < MAX_GUESSES:
        for key in frequent_molecules:
            smiles = smiles_lookup.get(key)
            if smiles and smiles not in seen:
                guesses.append(smiles)
                seen.add(smiles)
            if len(guesses) == MAX_GUESSES:
                break

    if not guesses:
        raise ValueError(f"No valid SMILES candidates for molecule_id={molecule_id}")
    submission_rows.append({"molecule_id": molecule_id, "smiles": ";".join(guesses)})

submission = pd.DataFrame(submission_rows, columns=["molecule_id", "smiles"])
output_path = Path("/kaggle/working/submission.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)

candidate_counts = submission["smiles"].str.split(";").map(len)
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert submission["molecule_id"].is_unique
assert submission["molecule_id"].tolist() == sample_submission["molecule_id"].astype(str).tolist()
assert submission["smiles"].notna().all() and submission["smiles"].str.len().gt(0).all()
assert candidate_counts.between(1, MAX_GUESSES).all()

print(f"\nSaved: {output_path}")
print(f"Rows: {len(submission):,} | unique IDs: {submission['molecule_id'].nunique():,}")
print(f"Guesses per row: min={candidate_counts.min()}, median={candidate_counts.median():.0f}, max={candidate_counts.max()}")
print(submission.head(5).to_string(index=False))